# Verify Connect 4 Model

This notebook verifies the performance of the trained PPO model by playing it against a Random Agent for 100 episodes.

In [ ]:
import sys
import os
import numpy as np
from stable_baselines3 import PPO
import random

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from pefforza.envs.connect4_env import Connect4Env

In [ ]:
def test_model(model_path, num_episodes=100):
    env = Connect4Env()
    
    if not os.path.exists(model_path):
        print(f"Model not found at {model_path}")
        return

    model = PPO.load(model_path)
    print(f"Loaded model from {model_path}")
    
    wins = 0
    losses = 0
    draws = 0
    
    print(f"Starting {num_episodes} games against Random Agent...")
    
    for i in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        # Toggle who starts? For PPO trained as Player 1, it expects to be Player 1.
        # If we want to test robustness, we could enforce Agent is always Player 1 for now.
        # Connect4Env default: Player 1 starts.
        
        while not done:
            # Player 1 (Agent) Turn
            # Model expects: 1=Self, 2=Opponent. Board is 1=P1, 2=P2.
            # Since Agent is P1, no swap needed.
            
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
            
            # Check validity for debug/fallback (Env handles invalid move by ignoring or ending? 
            # Connect4Env creates new state. If invalid, it might return large negative reward or stay same.
            # Let's trust model.predict gives valid if trained well, or Env handles it.
            
            obs, reward, terminated, truncated, info = env.step(action)
            
            if terminated or truncated:
                if reward == 1:
                    wins += 1
                else:
                    # Reward 0 usually means Draw in standard envs, but check implementation.
                    # If reward != 1 and terminated, likely draw or invalid move penalty?
                    # Assuming 1 = Win, 0 (or small) = Draw, -1 = Loss.
                    draws += 1
                done = True
                break
                
            # Player 2 (Random) Turn
            valid_moves = [c for c in range(env.cols) if env.board[0, c] == 0]
            if not valid_moves:
                draws += 1
                done = True
                break
                
            opp_action = random.choice(valid_moves)
            obs, reward, terminated, truncated, info = env.step(opp_action)
            
            if terminated or truncated:
                if reward == 1:
                     # Opponent (P2) won
                     losses += 1
                else:
                     draws += 1
                done = True
                break
    
    print("\nResults:")
    print(f"Wins: {wins}")
    print(f"Losses: {losses}")
    print(f"Draws: {draws}")
    win_rate = (wins / num_episodes) * 100
    print(f"Win Rate: {win_rate:.2f}%")
    
    return win_rate

In [ ]:
MODEL_PATH = os.path.join(project_root, "pefforza/agent/models/notebook_model.zip")
test_model(MODEL_PATH)